In [1]:
import json
import shutil
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
from IPython.display import display
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

In [ ]:
def load_scenes_for_selection(scenes_dir):
    scenes = []
    path = Path(scenes_dir)
    if not path.exists(): 
        return scenes
    for scene_dir in path.iterdir():
        if not scene_dir.is_dir(): continue
        ann_file = scene_dir / "annotation.json"
        if not ann_file.exists(): continue
        with open(ann_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            furnitures = [{"id": f["furniture_id"], "category": f["category"]} for f in data.get("furnitures", [])]
            scenes.append({
                "scene_id": data.get("scene_id", scene_dir.name),
                "furnitures": furnitures,
                "scene_dir": scene_dir
            })
    return scenes

def select_optimal_scenes(scenes_list, target_count=800, primary_category=None, max_furnitures=7):
    """
    Greedily selects scenes trying to:
    Filter out scenes with 8 or more furniture items.
    Maximize newly introduced, unrepresented categories.
    Maximize distinct categories present natively within the scene.
    Minimize duplicate furniture IDs.
    """
    selected_scenes = []
    seen_furniture = set()
    seen_categories = set()
    
    # Exclude scenes with lots of furniture
    remaining_scenes = [s for s in scenes_list if len(s['furnitures']) <= max_furnitures]
    
    target = min(target_count, len(remaining_scenes))
    print(f"Starting selection of {target} optimal scenes out of {len(remaining_scenes)} (filtered out large scenes)...")
    
    for step in range(target):
        best_scene_idx = -1
        # Set initial worst-possible score tuple. The size of this tuple depends on standard vs primary_category mode.
        if primary_category:
            best_score = (float('inf'), float('inf'), float('inf'), float('inf'), float('inf'))
        else:
            best_score = (float('inf'), float('inf'), float('inf'), float('inf'))
        
        for i, scene in enumerate(remaining_scenes):
            overlap = 0
            new_items = 0
            new_cats = 0
            
            # Determine distinct categories native to this scene
            scene_categories = set(f['category'].lower() for f in scene['furnitures'])
            scene_distinct_cat_count = len(scene_categories)
            
            # Determine how many globally completely *new* categories this introduces
            for cat in scene_categories:
                if cat not in seen_categories:
                    new_cats += 1
            
            primary_overlap = 0
            for f in scene['furnitures']:
                fid = f['id']
                if fid in seen_furniture:
                    overlap += 1
                    if primary_category and f['category'].lower() == primary_category.lower():
                        primary_overlap += 1
                else:
                    new_items += 1
            
            # The sorting criteria tuple (lower is better):
            if primary_category:
                # Minimize already-seen primary category items
                # Maximize newly introduced categories overall (-new_cats)
                # Maximize local distinct categories within the scene (-scene_distinct_cat_count)
                # Minimize overall overlaps
                # Maximize newly introduced items
                score = (primary_overlap, -new_cats, -scene_distinct_cat_count, overlap, -new_items)
            else:
                # Maximize newly introduced categories overall (-new_cats)
                # Maximize local distinct categories within the scene (-scene_distinct_cat_count)
                # Minimize overall overlaps
                # Maximize newly introduced items
                score = (-new_cats, -scene_distinct_cat_count, overlap, -new_items)
                
            if score < best_score:
                best_score = score
                best_scene_idx = i
        
        # Move the best scene to our selected list
        best_scene = remaining_scenes.pop(best_scene_idx)
        selected_scenes.append(best_scene)
        
        # Record its items and categories as seen
        for f in best_scene['furnitures']:
            seen_furniture.add(f['id'])
            seen_categories.add(f['category'].lower())
            
        if (step + 1) % 250 == 0:
            print(f" Selected {step + 1}/{target} scenes...")
            
    return selected_scenes

def analyze_subset(selected_scenes, name):
    """Generates and displays deep analytics for a selected subset of scenes."""
    total_furnitures = 0
    furniture_ids = []
    furniture_ids_by_cat = defaultdict(list)
    
    for scene in selected_scenes:
        for f in scene['furnitures']:
            fid = f['id']
            cat = f['category']
            total_furnitures += 1
            furniture_ids.append(fid)
            furniture_ids_by_cat[cat].append(fid)
            
    counter = Counter(furniture_ids)
    unique_f = len(counter)
    dups = sum(c - 1 for c in counter.values() if c > 1)
    dup_rate = (dups / total_furnitures * 100) if total_furnitures > 0 else 0
    
    print(f"=== {name} Subset Analysis ({len(selected_scenes)} scenes) ===")
    print(f"Total Furnitures:      {total_furnitures:,}")
    print(f"Unique Furnitures:     {unique_f:,}")
    print(f"Duplicate Instances:   {dups:,} ({(dup_rate):.2f}% Overall Dup Rate)")
    print("\nDistribution and Duplication Rate by Category:")
    
    cat_stats = []
    for cat, ids in furniture_ids_by_cat.items():
        cc = Counter(ids)
        tgt_total = len(ids)
        tgt_unique = len(cc)
        tgt_dups = sum(c - 1 for c in cc.values() if c > 1)
        tgt_dup_rate = (tgt_dups / tgt_total * 100) if tgt_total > 0 else 0
        cat_stats.append({
            "Category": cat,
            "Total Items": tgt_total,
            "Unique Items": tgt_unique,
            "Duplicates": tgt_dups,
            "Dup Rate %": round(tgt_dup_rate, 2)
        })
        
    df = pd.DataFrame(cat_stats).sort_values("Total Items", ascending=False)
    display(df)
    return df

all_bedrooms = load_scenes_for_selection("initial_process_bedrooms_scenes")

all_livings = load_scenes_for_selection("initial_process_living_room_scenes")

selected_bedrooms = select_optimal_scenes(all_bedrooms, target_count=800, primary_category='bed')

selected_livings = select_optimal_scenes(all_livings, target_count=800, primary_category='sofa')

df_bed_subset = analyze_subset(selected_bedrooms, "Optimal Bedrooms")
df_liv_subset = analyze_subset(selected_livings, "Optimal Living Rooms")

out_dir = Path("statistics")
out_dir.mkdir(exist_ok=True)

bed_out_path = out_dir / "selected_bedrooms.txt"
liv_out_path = out_dir / "selected_living_rooms.txt"

with open(bed_out_path, "w") as f:
    for s in selected_bedrooms:
        f.write(f"{s['scene_id']}\n")

with open(liv_out_path, "w") as f:
    for s in selected_livings:
        f.write(f"{s['scene_id']}\n")
        
print(f"\n✓ Saved selected bedroom scene IDs to: {bed_out_path}")
print(f"✓ Saved selected living room scene IDs to: {liv_out_path}")

In [ ]:
model_name = "openai/clip-vit-large-patch14"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP model on {device}...")
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
model.eval()

Loading CLIP model on cuda...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [ ]:
def get_clip_match(img_tensor, prompts_dict):
    all_prompts = []
    keys = list(prompts_dict.keys())
    lengths = []
    
    for k in keys:
        all_prompts.extend(prompts_dict[k])
        lengths.append(len(prompts_dict[k]))
        
    inputs = processor(text=all_prompts, images=img_tensor, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        image_emb = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_emb = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)
    
    sims = (image_emb @ text_emb.T).cpu().numpy()[0]
    
    scores = {}
    idx = 0
    for k, length in zip(keys, lengths):
        scores[k] = sims[idx:idx+length].mean()
        idx += length
        
    return max(scores, key=scores.get)

def classify_clip(img_path, room_type):
    img = Image.open(img_path).convert("RGB")
    
    storage_scale_prompts = {
        "small_storage": [
            "a small bedside nightstand", 
            "a tiny storage unit", 
            "a low side table with a drawer",
            "a small cabinet",
            "a minor storage element"
        ],
        "large_storage": [
            "a large massive storage piece", 
            "a tall closed wardrobe cabinet", 
            "a giant wall shelving unit", 
            "a large television media console",
            "a wide multi-drawer horizontal dresser"
        ]
    }
    
    return get_clip_match(img, storage_scale_prompts)


def process_and_save_scenes(selected_scenes, output_dir_name, room_type, prefix="deepfurn"):
    output_dir = Path(output_dir_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    scene_mapping = {}
    scene_index = 0
    total_furnitures_saved = 0
    
    print(f"\nCopying, classifying and saving {len(selected_scenes)} {room_type} scenes to '{output_dir}'...")

    for scene_info in tqdm(selected_scenes):
        old_scene_dir = Path(scene_info["scene_dir"])
        old_scene_id = old_scene_dir.name
        
        annotation_file = old_scene_dir / "annotation.json"
        if not annotation_file.exists(): continue
            
        with open(annotation_file, "r", encoding="utf-8") as f:
            original_annotation = json.load(f)
            
        new_scene_name = f"{prefix}_{scene_index:04d}"
        new_scene_dir = output_dir / new_scene_name
        new_scene_dir.mkdir(exist_ok=True)
        
        old_scene_img = old_scene_dir / "scene_image.jpg"
        new_scene_img = new_scene_dir / "scene_image.jpg"
        if old_scene_img.exists(): shutil.copy2(old_scene_img, new_scene_img)
            
        new_furniture_dir = new_scene_dir / "furnitures"
        new_furniture_dir.mkdir(exist_ok=True)
        
        filtered_furnitures = []
        seen_furniture_ids = set()
        
        for furn in original_annotation.get("furnitures", []):
            furniture_id = furn["furniture_id"]
            if furniture_id in seen_furniture_ids: continue
            seen_furniture_ids.add(furniture_id)
            
            img_filename = furn.get("image_filename", "")
            if img_filename:
                old_furn_img = old_scene_dir / img_filename
            else:
                old_furn_img = old_scene_dir / "furnitures" / f"{furniture_id}.jpg"
                if not old_furn_img.exists():
                    old_furn_img = old_scene_dir / f"{furniture_id}.jpg"
                    
            new_furn_img = new_furniture_dir / f"{furniture_id}.jpg"
            if old_furn_img.exists(): shutil.copy2(old_furn_img, new_furn_img)
            
            category = furn.get("category", "").lower()
            
            # Apply CLIP scaling logic strictly for ambiguous storage items
            if category in ["cabinet#shelf", "cabinet_shelf"]:
                if old_furn_img.exists():
                    category = classify_clip(old_furn_img, room_type)
                else: 
                    # Default failsafe if no image exists
                    category = "large_storage"
                
            filtered_furnitures.append({"furniture_id": f"{furniture_id}", "category": category})
            
        filtered_annotation = {"scene_name": new_scene_name, "furnitures": filtered_furnitures}
        
        new_annotation_file = new_scene_dir / "annotations.json"
        with open(new_annotation_file, "w", encoding="utf-8") as f:
            json.dump(filtered_annotation, f, indent=2)
            
        scene_mapping[old_scene_id] = {
            "new_name": new_scene_name,
            "index": scene_index,
            "original_furniture_count": len(original_annotation.get("furnitures", [])),
            "filtered_furniture_count": len(filtered_furnitures)
        }
        
        total_furnitures_saved += len(filtered_furnitures)
        scene_index += 1
        
    mapping_file = output_dir / "scene_mapping.json"
    with open(mapping_file, "w") as f:
        json.dump(scene_mapping, f, indent=2)
        
    print(f"Saved {scene_index} scenes and {total_furnitures_saved} furniture items to {output_dir}")

process_and_save_scenes(selected_bedrooms, "processed_bedrooms_scenes", "bedroom")
process_and_save_scenes(selected_livings, "processed_living_rooms_scenes", "living_room")

In [ ]:
def gather_images_by_category_per_room(base_dirs):
    # Returns: {dir_name: {category: {fid: img_path}}}
    room_imgs = defaultdict(lambda: defaultdict(dict))
    
    for base_dir in base_dirs:
        path = Path(base_dir)
        if not path.exists(): continue
        for scene_dir in path.iterdir():
            if not scene_dir.is_dir(): continue
            ann_file = scene_dir / "annotations.json"
            if not ann_file.exists(): continue
            
            with open(ann_file, "r", encoding="utf-8") as f:
                ann = json.load(f)
                
            furn_dir = scene_dir / "furnitures"
            for furn in ann.get("furnitures", []):
                cat = furn.get("category", "")
                fid = furn.get("furniture_id")
                img_path = furn_dir / f"{fid}.jpg"
                if img_path.exists():
                    room_imgs[base_dir][cat][fid] = img_path
    return room_imgs

def show_images(img_dict, title, max_imgs=40):
    img_paths = list(img_dict.values())
    if not img_paths:
        print(f"No images found for {title}")
        return
        
    n = min(len(img_paths), max_imgs)
    cols = 5
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(15, 3 * rows))
    for i, img_path in enumerate(img_paths[:n]):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(Image.open(img_path))
        plt.axis("off")
        plt.title(f"{img_path.name}")
    plt.suptitle(f"{title} (unique: {len(img_dict)})")
    plt.tight_layout()
    plt.show()

dirs_to_check = ["processed_bedrooms_scenes", "processed_living_rooms_scenes"]
room_imgs = gather_images_by_category_per_room(dirs_to_check)

new_categories = ["small_storage", "large_storage"]

for directory, category_imgs in room_imgs.items():
    print(f"\n{'='*80}\nShowing results for: {directory}\n{'='*80}")
    
    for cat in new_categories:
        if cat in category_imgs and len(category_imgs[cat]) > 0:
            show_images(
                category_imgs[cat], 
                f"Location: {directory} | Category: {cat.title()}", 
                max_imgs=200
            )